## Silver to Gold Transformation

This notebook builds the gold layer — business-ready metrics derived from the silver table.

**Why SQL for the gold layer?**
Both SQL and PySpark can accomplish everything in this notebook — as demonstrated in the
bronze_to_silver notebook where SQL equivalents are documented alongside the Python code.
SQL was chosen here because the gold layer consists primarily of aggregations and window
functions which are more concise and readable in SQL. PySpark adds value when you need
programmatic control, reusable functions, or complex multi-step transformations — which
is why it was used as the primary approach in the bronze to silver layer.

In [0]:
SELECT * FROM stock_pipeline.silver.stock_prices
LIMIT 5

## Daily Prices Table

Enriches the silver stock data with daily percentage return for each ticker.
This is the foundation table for all price-based analysis in the gold layer.

**Columns:**
- `Date` — trading date
- `Ticker` — stock symbol
- `Sector` — sector classification
- `Open` — opening price
- `High` — highest price of the day
- `Low` — lowest price of the day
- `Close` — closing price
- `Volume` — shares traded
- `Daily_Return_Pct` — percentage change from previous day's close. NULL for the first trading day of each ticker
- daily % change = ((today's close - yesterday's close) / yesterday's close) * 100

In [0]:
SELECT *,
    ROUND(
        ((Close - LAG(Close, 1) OVER (PARTITION BY Ticker ORDER BY Date)) /
         LAG(Close, 1) OVER (PARTITION BY Ticker ORDER BY Date)) * 100
    , 2) AS Daily_Return_Pct
FROM stock_pipeline.silver.stock_prices

In [0]:
CREATE OR REPLACE TABLE stock_pipeline.gold.daily_prices AS
SELECT *,
    ROUND(
        ((Close - LAG(Close, 1) OVER (PARTITION BY Ticker ORDER BY Date)) /
         LAG(Close, 1) OVER (PARTITION BY Ticker ORDER BY Date)) * 100
    , 2) AS Daily_Return_Pct
FROM stock_pipeline.silver.stock_prices

In [0]:
SELECT * FROM stock_pipeline.gold.daily_prices
ORDER BY Ticker, Date
LIMIT 20

## Moving Averages Table

Calculates 7-day and 30-day rolling averages of closing prices for each ticker.
Used to identify price trends — if Close is above MA_30_Day the stock is in an uptrend.

**Columns:**
- `Date` — trading date
- `Ticker` — stock symbol
- `Sector` — sector classification
- `Close` — closing price
- `MA_7_Day` — average closing price over last 7 trading days
- `MA_30_Day` — average closing price over last 30 trading days

In [0]:
SELECT
    Date,
    Ticker,
    Sector,
    Close,
    ROUND(AVG(Close) OVER (
        PARTITION BY Ticker
        ORDER BY Date
        ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
    ), 2) AS MA_7_Day,
    ROUND(AVG(Close) OVER (
        PARTITION BY Ticker
        ORDER BY Date
        ROWS BETWEEN 29 PRECEDING AND CURRENT ROW
    ), 2) AS MA_30_Day
FROM stock_pipeline.silver.stock_prices

In [0]:
CREATE OR REPLACE TABLE stock_pipeline.gold.moving_averages AS
SELECT
    Date,
    Ticker,
    Sector,
    Close,
    ROUND(AVG(Close) OVER (
        PARTITION BY Ticker
        ORDER BY Date
        ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
    ), 2) AS MA_7_Day,
    ROUND(AVG(Close) OVER (
        PARTITION BY Ticker
        ORDER BY Date
        ROWS BETWEEN 29 PRECEDING AND CURRENT ROW
    ), 2) AS MA_30_Day
FROM stock_pipeline.silver.stock_prices

## Sector Summary Table

Aggregates individual stock data to the sector level per day.
Useful for comparing sector performance over time.

**Columns:**
- `Date` — trading date
- `Sector` — sector name (FAANG, Finance, Healthcare etc)
- `Num_Stocks` — number of stocks in the sector
- `Avg_Close` — average closing price across all stocks in the sector
- `Avg_Daily_Return_Pct` — average daily % return across the sector
- `Total_Volume` — total shares traded across all stocks in the sector

In [0]:
SELECT
    Date,
    Sector,
    COUNT(DISTINCT Ticker)        AS Num_Stocks,
    ROUND(AVG(Close), 2)          AS Avg_Close,
    ROUND(AVG(Daily_Return_Pct), 2) AS Avg_Daily_Return_Pct,
    SUM(Volume)                   AS Total_Volume
FROM stock_pipeline.gold.daily_prices
GROUP BY Date, Sector
ORDER BY Date, Sector
LIMIT 20

In [0]:
CREATE OR REPLACE TABLE stock_pipeline.gold.sector_summary AS
SELECT
    Date,
    Sector,
    COUNT(DISTINCT Ticker)          AS Num_Stocks,
    ROUND(AVG(Close), 2)            AS Avg_Close,
    ROUND(AVG(Daily_Return_Pct), 2) AS Avg_Daily_Return_Pct,
    SUM(Volume)                     AS Total_Volume
FROM stock_pipeline.gold.daily_prices
GROUP BY Date, Sector
ORDER BY Date, Sector

## Volatility Table

Measures the risk/price fluctuation of each stock using 30-day rolling standard deviation of daily returns.
A higher volatility means the stock price swings more dramatically — higher risk, higher potential reward.
Used by risk teams, hedge funds, and traders to assess and compare stock risk profiles.

**Columns:**
- `Date` — trading date
- `Ticker` — stock symbol
- `Sector` — sector classification
- `Close` — closing price
- `Avg_30Day_Return` — average daily % return over the last 30 trading days
- `Volatility_30Day` — standard deviation of daily % returns over the last 30 trading days. Higher = more volatile/risky

In [0]:
SELECT
    Date,
    Ticker,
    Sector,
    Close,
    ROUND(AVG(Daily_Return_Pct) OVER (
        PARTITION BY Ticker
        ORDER BY Date
        ROWS BETWEEN 29 PRECEDING AND CURRENT ROW
    ), 2) AS Avg_30Day_Return,
    ROUND(STDDEV(Daily_Return_Pct) OVER (
        PARTITION BY Ticker
        ORDER BY Date
        ROWS BETWEEN 29 PRECEDING AND CURRENT ROW
    ), 2) AS Volatility_30Day
FROM stock_pipeline.gold.daily_prices
ORDER BY Ticker, Date
LIMIT 20

In [0]:
CREATE OR REPLACE TABLE stock_pipeline.gold.volatility AS
SELECT
    Date,
    Ticker,
    Sector,
    Close,
    ROUND(AVG(Daily_Return_Pct) OVER (
        PARTITION BY Ticker
        ORDER BY Date
        ROWS BETWEEN 29 PRECEDING AND CURRENT ROW
    ), 2) AS Avg_30Day_Return,
    ROUND(STDDEV(Daily_Return_Pct) OVER (
        PARTITION BY Ticker
        ORDER BY Date
        ROWS BETWEEN 29 PRECEDING AND CURRENT ROW
    ), 2) AS Volatility_30Day
FROM stock_pipeline.gold.daily_prices

## 52-Week High/Low Table

Shows the highest and lowest closing prices for each stock over the past year,
along with how far the current price is from those extremes.
Every major financial platform (Bloomberg, Yahoo Finance, CNBC) displays this metric.
Useful for identifying stocks trading near yearly highs or lows.

**Columns:**
- `Date` — trading date
- `Ticker` — stock symbol
- `Sector` — sector classification
- `Close` — current closing price
- `High_52_Week` — highest closing price over the past 252 trading days
- `Low_52_Week` — lowest closing price over the past 252 trading days
- `Pct_From_High` — how far current price is below the 52-week high (negative = below high)
- `Pct_From_Low` — how far current price is above the 52-week low (positive = above low)

In [0]:
SELECT
    Date,
    Ticker,
    Sector,
    Close,
    MAX(Close) OVER (
        PARTITION BY Ticker
        ORDER BY Date
        ROWS BETWEEN 251 PRECEDING AND CURRENT ROW
    ) AS High_52_Week,
    MIN(Close) OVER (
        PARTITION BY Ticker
        ORDER BY Date
        ROWS BETWEEN 251 PRECEDING AND CURRENT ROW
    ) AS Low_52_Week,
    ROUND(((Close - MAX(Close) OVER (
        PARTITION BY Ticker ORDER BY Date
        ROWS BETWEEN 251 PRECEDING AND CURRENT ROW)) /
        MAX(Close) OVER (
        PARTITION BY Ticker ORDER BY Date
        ROWS BETWEEN 251 PRECEDING AND CURRENT ROW)) * 100, 2) AS Pct_From_High,
    ROUND(((Close - MIN(Close) OVER (
        PARTITION BY Ticker ORDER BY Date
        ROWS BETWEEN 251 PRECEDING AND CURRENT ROW)) /
        MIN(Close) OVER (
        PARTITION BY Ticker ORDER BY Date
        ROWS BETWEEN 251 PRECEDING AND CURRENT ROW)) * 100, 2) AS Pct_From_Low
FROM stock_pipeline.silver.stock_prices
ORDER BY Ticker, Date
LIMIT 20

In [0]:
CREATE OR REPLACE TABLE stock_pipeline.gold.fifty_two_week AS
SELECT
    Date,
    Ticker,
    Sector,
    Close,
    MAX(Close) OVER (
        PARTITION BY Ticker
        ORDER BY Date
        ROWS BETWEEN 251 PRECEDING AND CURRENT ROW
    ) AS High_52_Week,
    MIN(Close) OVER (
        PARTITION BY Ticker
        ORDER BY Date
        ROWS BETWEEN 251 PRECEDING AND CURRENT ROW
    ) AS Low_52_Week,
    ROUND(((Close - MAX(Close) OVER (
        PARTITION BY Ticker ORDER BY Date
        ROWS BETWEEN 251 PRECEDING AND CURRENT ROW)) /
        MAX(Close) OVER (
        PARTITION BY Ticker ORDER BY Date
        ROWS BETWEEN 251 PRECEDING AND CURRENT ROW)) * 100, 2) AS Pct_From_High,
    ROUND(((Close - MIN(Close) OVER (
        PARTITION BY Ticker ORDER BY Date
        ROWS BETWEEN 251 PRECEDING AND CURRENT ROW)) /
        MIN(Close) OVER (
        PARTITION BY Ticker ORDER BY Date
        ROWS BETWEEN 251 PRECEDING AND CURRENT ROW)) * 100, 2) AS Pct_From_Low
FROM stock_pipeline.silver.stock_prices

## Volume Spikes Table

Detects abnormal trading activity by flagging days where volume exceeds 2x the 30-day average.
Unusual volume often precedes or accompanies major price moves, earnings surprises, or news events.
This type of anomaly detection is directly applicable to fraud detection, risk monitoring,
and algorithmic trading systems.

**Columns:**
- `Date` — trading date
- `Ticker` — stock symbol
- `Sector` — sector classification
- `Volume` — actual shares traded that day
- `Avg_30Day_Volume` — average daily volume over the last 30 trading days
- `Volume_Ratio` — how many times the current volume exceeds the 30-day average
- `Is_Spike` — true if volume is more than 2x the 30-day average

In [0]:
SELECT
    Date,
    Ticker,
    Sector,
    Volume,
    ROUND(AVG(Volume) OVER (
        PARTITION BY Ticker
        ORDER BY Date
        ROWS BETWEEN 29 PRECEDING AND CURRENT ROW
    )) AS Avg_30Day_Volume,
    ROUND(Volume / AVG(Volume) OVER (
        PARTITION BY Ticker
        ORDER BY Date
        ROWS BETWEEN 29 PRECEDING AND CURRENT ROW
    ), 2) AS Volume_Ratio,
    CASE
        WHEN Volume >= 2 * AVG(Volume) OVER (
            PARTITION BY Ticker
            ORDER BY Date
            ROWS BETWEEN 29 PRECEDING AND CURRENT ROW
        ) THEN true
        ELSE false
    END AS Is_Spike
FROM stock_pipeline.silver.stock_prices
ORDER BY Volume_Ratio DESC
LIMIT 20

In [0]:
CREATE OR REPLACE TABLE stock_pipeline.gold.volume_spikes AS
SELECT
    Date,
    Ticker,
    Sector,
    Volume,
    ROUND(AVG(Volume) OVER (
        PARTITION BY Ticker
        ORDER BY Date
        ROWS BETWEEN 29 PRECEDING AND CURRENT ROW
    )) AS Avg_30Day_Volume,
    ROUND(Volume / AVG(Volume) OVER (
        PARTITION BY Ticker
        ORDER BY Date
        ROWS BETWEEN 29 PRECEDING AND CURRENT ROW
    ), 2) AS Volume_Ratio,
    CASE
        WHEN Volume >= 2 * AVG(Volume) OVER (
            PARTITION BY Ticker
            ORDER BY Date
            ROWS BETWEEN 29 PRECEDING AND CURRENT ROW
        ) THEN true
        ELSE false
    END AS Is_Spike
FROM stock_pipeline.silver.stock_prices

In [0]:
SHOW TABLES IN stock_pipeline.gold

In [0]:
SELECT 'daily_prices' AS table_name, COUNT(*) AS row_count FROM stock_pipeline.gold.daily_prices
UNION ALL
SELECT 'moving_averages', COUNT(*) FROM stock_pipeline.gold.moving_averages
UNION ALL
SELECT 'sector_summary', COUNT(*) FROM stock_pipeline.gold.sector_summary
UNION ALL
SELECT 'volatility', COUNT(*) FROM stock_pipeline.gold.volatility
UNION ALL
SELECT 'fifty_two_week', COUNT(*) FROM stock_pipeline.gold.fifty_two_week
UNION ALL
SELECT 'volume_spikes', COUNT(*) FROM stock_pipeline.gold.volume_spikes